In [17]:
from sympy.printing.pretty.pretty_symbology import line_width
# auto-reload changed source files when they are imported
%reload_ext autoreload
%autoreload 2

# add top repo dir to path so that src can be imported
import sys
sys.path.append("..")

%matplotlib inline
from src.data import load_raman_spectra, load_crystal_structures
import numpy as np
from collections import Counter, defaultdict
import torch_geometric as pyg
import codecs
import torch
from src.visualization import plotting
import pandas as pd

In [5]:
def filter_graphs(data_lst, wl_lst):
    valid = []
    validw = []
    removed = 0
    for i, d in enumerate(data_lst):
        if d.edge_index is not None and d.edge_index.size(1) != 0:
            valid.append(data_lst[i])
            validw.append(wl_lst[i])
            continue
        else:
            removed += 1
        # Проверка dist
        if hasattr(d,"dist") and d.dist is not None:
            valid.append(data_lst[i])
            validw.append(wl_lst[i])
            continue
        else:
            removed += 1
    return valid, validw, removed

In [6]:
cif_mineral_names = []
cif_graphs = []
big_cif_file_path = '../data/raw/cifdata_10.txt'
temp_file_path = '../data/raw/tempcif.txt'
cur_mineral_name = ''
cur_cif_lines = []
with open(big_cif_file_path,'r',errors='ignore') as f:
    for line in f:
        if line.startswith("_chemical_formula_sum ''"):
            continue
        if line.startswith('_chemical_name_mineral'):
            cur_mineral_name = line.split("'")[1]
        if line.startswith('_amcsd_formula_title '):
            cur_mineral_name = line.split("'")[1]
        if line.startswith('END'):
            try:
                with codecs.open(temp_file_path,'w','utf-8') as out:
                    out.write(''.join(cur_cif_lines))
                _,G = load_crystal_structures.load_single_crystal_structure(
                                                    temp_file_path,
                                                    min_distance_for_edge=-1,
                                                    max_distance_for_edge=10,
                                                    )
                cif_graphs.append(G)
                cif_mineral_names.append(cur_mineral_name)
            except:
                pass
            cur_cif_lines = []
        else:
            cur_cif_lines.append(line)

In [18]:
model_wavenumber_values = np.load('../data/processed/wavenumber_vals_v3.npy')
data_list = []
wavelengths = []
for i in {514,532,780,785}:
    print("current wavelength: ",i)
    raman_file_paths, raman_mineral_names, raman_spectra, raman_wavelengths = load_raman_spectra.load_raman_data(model_wavenumber_values,wavelength=i)
    print(len(raman_mineral_names), "Raman spectra loaded, each of length", len(raman_spectra[0]))
    all_minerals = set(cif_mineral_names+raman_mineral_names)
    cif_counter = Counter(cif_mineral_names)
    raman_counter = Counter(raman_mineral_names)
    minimum_number_for_each = 1
    minerals_for_dataset = []
    for mineral in all_minerals:
        if cif_counter[mineral] >= minimum_number_for_each and raman_counter[mineral] >= minimum_number_for_each:
            minerals_for_dataset.append(mineral)
    try:
        minerals_for_dataset.remove('Diamond')
    except:
        print("no diamond at ",i)
    try:
        minerals_for_dataset.remove('Sulphur')
    except:
        print("no sulphur at ",i)
    try:
        minerals_for_dataset.remove('Silicon')
    except:
        print("no silicon at ",i)
    print(len(minerals_for_dataset),'/',len(all_minerals))
    for mineral in minerals_for_dataset:
        cur_mineral_raman_indices = [i for i, x in enumerate(raman_mineral_names) if x == mineral]
        cur_mineral_graph_indices = [i for i, x in enumerate(cif_mineral_names) if x == mineral]
        for i_raman, i_graph in zip(cur_mineral_raman_indices,cur_mineral_graph_indices):
            cur_graph = pyg.utils.convert.from_networkx(cif_graphs[i_graph])
            cur_graph['y'] = raman_spectra[i_raman]
            wavelengths.append(i/1e2)
            cur_graph['mineral'] = mineral
            data_list.append(cur_graph)
    data_list, wavelengths, removed = filter_graphs(data_list,wavelengths)
    print(f'left: {len(data_list)} wl: {len(wavelengths)}, removed: {removed}')
for g, wl in zip(data_list, wavelengths): g.wl = torch.tensor([wl], dtype=torch.float32)
print((pd.DataFrame(wavelengths)).value_counts())

current wavelength:  785
2208 Raman spectra loaded, each of length 266
no silicon in  785
668 / 8218
left: 1233 wl: 1233, removed: 48
current wavelength:  514
12311 Raman spectra loaded, each of length 266
295 / 7993
left: 4108 wl: 4108, removed: 4
current wavelength:  532
7149 Raman spectra loaded, each of length 266
1470 / 8658
left: 7545 wl: 7545, removed: 82
current wavelength:  780
runtime warning at  ../data/raw/raman\unrated_unoriented\DecrespignyiteY__R060302__Raman__780__0__unoriented__Raman_Data_RAW__38764.txt
4550 Raman spectra loaded, each of length 266
1076 / 8405
left: 9941 wl: 9941, removed: 44
0   
5.32    3437
5.14    2875
7.80    2396
7.85    1233
Name: count, dtype: int64


In [26]:
save_path = '../data/processed/v6.pt'
data, slices, _ = pyg.data.collate.collate(
    data_list[0].__class__,
    data_list=data_list,
    increment=False,
    add_batch=True,
)
data.x = data.x.type(torch.FloatTensor)
data.wl = torch.tensor(wavelengths, dtype=torch.float32)
data.pos = data.pos.type(torch.FloatTensor)
data.dist = data.dist.type(torch.FloatTensor)
data.y = torch.stack([torch.Tensor(yi) for yi in data.y]).type(torch.FloatTensor)

collated_data = (data, slices)
torch.save(collated_data, save_path)